In [ ]:
# standard libraries
import numpy as np
import matplotlib.pyplot as plt
# use display and Image to update plot in loop
from IPython import display
from PIL import Image
# the pytorch 
import torch

torch.set_default_dtype(torch.float64)

# Parameters and exact solution for reference

In [ ]:
def oscillator(d, w0, t): 
    '''
    Exact solutions of the damped harmonic oscillator:

    d²x/dt² + 2d.v + k.x = 0

    Using initial conditions x(0) = 1, v(0) = 0.

    Parameters
    ----------
    d: float
        The damping term of the ODE above.
    w0: float
        The natural frequency, k = w0**2
    t: float or list
        The time instant
    
    Returns
    -------
    (tuple) Solutions x(t) and v(t)

    '''
    # assert to the underdamped case
    assert d < w0
    # the damped frequency
    w = np.sqrt(w0**2-d**2)
    # phase that satisfy initial conditions
    phi = np.arctan(-d/w)
    # x(0) = 1 implies A.cos(phi) = 1
    A = 1/np.cos(phi)
    # the oscillating terms
    cos = np.cos(phi+w*t)
    sin = np.sin(phi+w*t)
    # the damping term
    exp = np.exp(-d*t)
    # the solution x(t) and v(t)
    x =  A*exp*cos
    v = -A*exp*(d*cos+w*sin)
    # return the solutions
    return x, v

In [ ]:
# PARAMETERS
mu = 4 # damping and 
w0 = 20 # natural frequency
tmax = 1

# label 'ref' as the reference solution
tref = np.linspace(0, tmax, 100)
xref, vref = oscillator(mu/2, w0, tref)

# PLOTS the exact solution
plt.figure(figsize=(8,2), facecolor='white')
plt.plot(tref, xref)
plt.axis("off")
plt.show()

# The Neural Nework class

In [ ]:
import torch.nn as nn
# THE FCN class inherits from nn.Module
class FCN(nn.Module):
    '''
    Define a simple fully connected neural network.

    Parameters
    ----------
    N_INPUT: int
        Size of input layer
    N_OUTPUT: int
        Size of the output layer
    N_HIDDEN: int
        Size of each hidden layer
    N_LAYERS: int
        Number of hidden layers

    '''
    def __init__(self, N_INPUT, N_OUTPUT, N_HIDDEN, N_LAYERS):
        '''
        Init the NN using the Tanh activation for all layers.
        '''
        # calls init from nn.Module
        super().__init__() 

        # defines the activation function
        activation = nn.ELU
        layers = [nn.Linear(N_INPUT, N_HIDDEN), activation()]
        for _ in range(N_LAYERS-1):
            layers.append(nn.Sequential(nn.Conv1d(1, 1, N_HIDDEN, N_HIDDEN), activation()))
            layers.append(nn.Dropout(p=0.05))
        layers.append(nn.Linear(N_HIDDEN, N_OUTPUT))
        # sequential defines the sequence of calls for each term
        self.the_nn = nn.Sequential(*layers)

    def forward(self, x):
        '''
        Forward method from input to output.

        Parameters
        ----------
        x: the input data

        Returns
        -------
        y: the output of the last layer
        '''
        y = self.the_nn(x)
        return y

# Define the training function

In [ ]:
def training(x0, v0, ts, batch_size=None, max_epochs=50000, n_hidden=3, s_hidden=32, seed=None, plotter=lambda *x: None):
    '''
    INPUT:
        x0, v0: initial conditions
        ts: array with t points for training
        batch_size: number of t-elements per iteration, if None, use all
        max_epochs: max number of training iterations
        n_hidden: number of hidden layers
        s_hidden: size of each hidden layer
        seed: random numbers seed, if None, does not apply manual seed
        plotter: function to call for plots, default: empty
    '''
    
    # fix the seed for testing only
    if seed is not None:
        torch.manual_seed(seed)
    
    # create NN
    # input = 1 neuron (time)
    # output = 2 neurons (x, v)
    model = FCN(1, 2, s_hidden, n_hidden)
    
    # choose optimizer method: Adam
    # see: https://arxiv.org/abs/1412.6980
    #      well suited for problems that are 
    #      large in terms of data and/or parameters
    # learing rate of 1e-3 is the default
    # ----------------------------------------------
    # QUESTION: how does pytorch identify the 
    #           parameters in our nn construction?
    # ----------------------------------------------
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # define batch_size and auxiliary lists
    if batch_size == None:
        batch_size = len(ts) - 1 # exclude 0
    # list of all points, excluding zero
    #     this will be used to sample points later, 
    #     and zero will be added by hand always!
    all_points = np.arange(1, len(ts)) 
    # convert ts list to torch Tensor with column vector shape
    # --------------------------------------------------------
    # QUESTION: I still don't understand the autograd process
    #           What's the meaning of requires_grad?
    #           What is pytorch doing underneath?
    # --------------------------------------------------------
    ts = torch.from_numpy(ts).view(-1,1).requires_grad_(True)
    
    # array to store loss vs epoch
    loss_array = []
    
    # run training iterations/epochs
    for i in range(max_epochs):
        # --------------------------------------------------------
        # QUESTION: again, I don' understand the autograd yet
        # --------------------------------------------------------
        optimizer.zero_grad()
    
        # batch sampling: zero plus random choice
        t_sample = [0] + list(np.sort(np.random.choice(all_points, batch_size, False)))
        tt = ts[t_sample]

        # Run the model to calculate the current prediction
        yh = model(tt)
        # extracts the x(t) and v(t)
        xh = yh[:,0].view(-1,1) # view reshapes to column form
        vh = yh[:,1].view(-1,1)

        # calculates the derivative: dx/dt
        dxdt = torch.autograd.grad(xh, tt, grad_outputs=torch.ones_like(xh), create_graph=True)[0]
        # calculates the derivative: dv/dt
        dvdt = torch.autograd.grad(vh, tt, grad_outputs=torch.ones_like(vh), create_graph=True)[0]
        # --------------------------------------------------------
        # QUESTION: again, I don' understand the autograd yet
        #           what is grad_outputs?
        #           what does create_graph?
        #           how is it calculating derivatives? 
        #                   Forwad, backward, symmetric?
        # --------------------------------------------------------

        # QUANTITIES TO CALCULATE LOSSES
        # rescale by 1/w0 or 1/w0**2 to fix units
        # and make sure all residues have the same dimensions/units
        edo1 = (dxdt - vh) / w0
        edo2 = (dvdt + mu*vh + w0**2*xh) / w0**2
        deltax = (xh[0]-x0)
        deltav = (vh[0]-v0) / w0
        # compose the loss function from the quantities above
        loss = torch.mean(edo1**2)
        loss += torch.mean(edo2**2)
        loss += 0.1 * torch.mean(deltax**2)
        loss += 0.1 * torch.mean(deltav**2)
        # store into array
        loss_array += [loss.detach().numpy()]
    
        # run the backward propagation and optimization step
        loss.backward()
        optimizer.step()
        
        # call plotter
        plotter(model, ts, i, loss_array)
        
    # return trained model
    return model

# Define the plot function

In [ ]:
def plot_data(tref, xref, model, loss, ts, epoch, files, steps_gif=100, steps_jupyter=1000):
    '''
    INPUT:
        tref, xref: the reference solution
        model: current model
        ts: t points in pytorch notation
        epoch: current iteration
        files: output list with file names for gif later
        steps: plot only after some steps
    '''
    # quit if not in the plot epoch
    if (epoch+1) % steps_gif != 0:
        return 0

    # rerun prediction with all reference points
    ypred = model(ts)
    xpred = ypred[:,0].detach().numpy() # detach is again a autograd requirement...
    vpred = ypred[:,1].detach().numpy()
        
    # file name to save
    file = "pinn_%.8i.png"%(epoch+1)
    # store into list of files
    files.append(file)
    
    # --------------------------------------------------------------
    # PLOTTING
    # --------------------------------------------------------------
    # init the figure
    plt.figure(figsize=(8,6), facecolor='white') 
    
    # plot solutions
    plt.subplot(211)
    # plot the exact solution
    plt.plot(tref, xref, color="grey", linewidth=2, alpha=0.8, label="Exact solution")
    # plot the NN generated data
    plt.plot(tref, xpred, 'o-', color="tab:blue", linewidth=4, alpha=0.8, label="Neural network prediction")
    # defines a legend
    l = plt.legend(loc=(1.01,0.34), frameon=False, fontsize="large")
    # fix ranges to make animation stable
    plt.xlim(-0.05, 1.05)
    plt.ylim(-1.10, 1.10)
    # title as epoch label
    plt.text(1.065, 0.7, "Training step: %i"%(epoch+1), fontsize="xx-large", color="k")
    plt.text(1.065, 0.4, "Loss: %f"%(loss[-1]), fontsize="xx-large", color="k")
    # remove axis so it looks 'cool'
    plt.axis("off")
    
    # plot error
    plt.subplot(212)
    plt.loglog(loss, '-')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    
    # save to outputfile
    plt.savefig(file, bbox_inches='tight', pad_inches=0.1, dpi=100, facecolor="white")
    # close figure
    plt.close()
    # --------------------------------------------------------------
    
    # update jupyter display only every steps_jupyter epochs
    if (epoch+1) % steps_jupyter == 0:
        # clear only once new data is available
        display.clear_output(wait=True)
        # display new data
        display.display(Image.open(file))
    
    # just to mark the end
    return 0

# Run it all!

In [ ]:
files = [] # init empty, plotter will fill it
# define plotter function
plotter = lambda model, ts, epoch, loss: plot_data(tref, xref, model, loss, ts, epoch, files, steps_gif=100, steps_jupyter=1000)

# extract initial conditions from reference data
x0 = xref[0]
v0 = vref[0]
# run model
final_model = training(x0, v0, tref, batch_size=2, max_epochs=50000, n_hidden=5, s_hidden=64, seed=123, plotter=plotter)

# Animate solution

In [ ]:
def save_gif_PIL(outfile, files, fps=5, loop=0):
    '''
    Creates a gif to outfile from the images in files.

    Parameters
    ----------
    outfile: str
        name of the output gif file

    files: list of str
        list with the name of the image files

    fps: int
        frames per second defines the duration as 1000/fps [ms]
    
    loop: bool
        if True, gif loops at the end
    '''
    # define list of images
    imgs = [Image.open(file) for file in files]
    # call save gif to outfile
    imgs[0].save(fp=outfile,
                 format='GIF', 
                 append_images=imgs[1:], 
                 save_all=True, 
                 duration=int(1000/fps),
                 loop=loop)

In [ ]:
# once everything is finished, call function to save gif
save_gif_PIL("pinn_batch_010.gif", files, fps=20, loop=0)